Demo of Caltech101 Step2

In [ ]:
import os
import torch
import timm

# 1. Configuration & Paths
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = '/content/drive/MyDrive/caltech101_vit_augreg_best.pth'

# 2. Define Architecture (Skeleton)
print("Building model architecture...")
image_model = timm.create_model('vit_small_patch16_224.augreg_in21k_ft_in1k', pretrained=True).to(device)
image_model.reset_classifier(0)
projection = torch.nn.Linear(384, 768).to(device)
logit_scale = torch.nn.Parameter(torch.tensor(0.0, device=device))

# 3. Check for existing weights
if os.path.exists(save_path):
    print(f"📦 Found saved weights at {save_path}. Loading...")

    checkpoint = torch.load(save_path, map_location=device)

    # Load the "brain" into the skeleton
    image_model.load_state_dict(checkpoint['image_model_state_dict'])
    projection.load_state_dict(checkpoint['projection_state_dict'])
    logit_scale.data = checkpoint['logit_scale']
    classes = checkpoint['classes']

    image_model.eval()
    projection.eval()
    print(f"✅ Model loaded successfully! (Previous Accuracy: {checkpoint.get('accuracy', 0):.2f}%)")
    print("🚀 You can skip training and go straight to the Demo!")

    SHOULD_TRAIN = False
else:
    print("❌ No saved weights found. Proceeding to training mode...")
    SHOULD_TRAIN = True

# 4. Conditional Training
if SHOULD_TRAIN:
    # ... (Insert your training loop code here) ...
    print("Training complete.")

Building model architecture...
📦 Found saved weights at /content/drive/MyDrive/caltech101_vit_augreg_best.pth. Loading...
✅ Model loaded successfully! (Previous Accuracy: 95.62%)
🚀 You can skip training and go straight to the Demo!


In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from transformers import BertTokenizer, BertModel
from tqdm.auto import tqdm
import timm
import gradio as gr
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. DATA PREPARATION (CALTECH-101)
# ==========================================
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.Lambda(lambda x: x.convert('RGB') if x.mode != 'RGB' else x),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # ViT standard
])

full_dataset = torchvision.datasets.Caltech101(root='./data', download=True, transform=transform)
classes = [c.replace('_', ' ') for c in full_dataset.categories]

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_set, test_set = random_split(full_dataset, [train_size, test_size])

# Using num_workers=0 to avoid the "AssertionError: child process" error in Colab
trainloader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
testloader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=0)

# ==========================================
# 3. TEXT ENCODER (BERT)
# ==========================================
print("Encoding text labels with BERT...")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

def encode_texts(prompts):
    inputs = tokenizer(prompts, padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return F.normalize(outputs.last_hidden_state[:, 0, :], dim=-1)

text_embeddings = encode_texts([f"a photo of a {c}" for c in classes])

# Clean up BERT to save VRAM
del bert_model
torch.cuda.empty_cache()

# ==========================================
# 4. IMAGE ENCODER (ViT AUGREG)
# ==========================================
# Model: vit_small_patch16_224.augreg_in21k_ft_in1k
# Feature dim: 384
image_model = timm.create_model('vit_small_patch16_224.augreg_in21k_ft_in1k', pretrained=True).to(device)
image_model.reset_classifier(0) # Remove the 1k classification head

# --- UNFREEZE BACKBONE ---
for param in image_model.parameters():
    param.requires_grad = True

# Projection Layer (384 -> 768)
projection = nn.Linear(384, 768).to(device)

# Learnable Temperature
logit_scale = nn.Parameter(torch.tensor(np.log(1 / 0.07), device=device))

# Optimizer
optimizer = torch.optim.AdamW([
    {'params': image_model.parameters(), 'lr': 1e-5},
    {'params': projection.parameters(), 'lr': 5e-5},
    {'params': [logit_scale], 'lr': 5e-5}
], weight_decay=0.01)

scaler = torch.amp.GradScaler('cuda')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Encoding text labels with BERT...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_214/2418468430.py:89: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler('cuda')


In [ ]:
# ==========================================
# 5. TRAINING LOOP
# ==========================================
epochs = 10
best_acc = 0.0

for epoch in range(epochs):
    image_model.train(); projection.train()
    total_loss = 0

    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            features = image_model(images)
            image_features = F.normalize(projection(features), dim=-1)
            logits = (logit_scale.exp() * image_features @ text_embeddings.T)
            loss = F.cross_entropy(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    # Evaluation
    image_model.eval(); projection.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            features = image_model(images)
            image_features = F.normalize(projection(features), dim=-1)
            preds = (image_features @ text_embeddings.T).argmax(dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    print(f"✅ Epoch {epoch+1} Test Accuracy: {acc:.2f}%")

    # SAVE BEST MODEL
    if acc > best_acc:
        best_acc = acc
        torch.save({
            'image_model_state_dict': image_model.state_dict(),
            'projection_state_dict': projection.state_dict(),
            'logit_scale': logit_scale.data,
            'classes': classes,
            'accuracy': best_acc
        }, '/content/drive/MyDrive/caltech101_vit_augreg_best.pth')
        print("💾 New Best Weights Saved!")



Epoch 1/10:   0%|          | 0/217 [00:00<?, ?it/s]

✅ Epoch 1 Test Accuracy: 53.40%
💾 New Best Weights Saved!


Epoch 2/10:   0%|          | 0/217 [00:00<?, ?it/s]

✅ Epoch 2 Test Accuracy: 67.40%
💾 New Best Weights Saved!


Epoch 3/10:   0%|          | 0/217 [00:00<?, ?it/s]

...

✅ Epoch 8 Test Accuracy: 95.22%
💾 New Best Weights Saved!


Epoch 9/10:   0%|          | 0/217 [00:00<?, ?it/s]

✅ Epoch 9 Test Accuracy: 94.99%


Epoch 10/10:   0%|          | 0/217 [00:00<?, ?it/s]

✅ Epoch 10 Test Accuracy: 95.62%
💾 New Best Weights Saved!


In [ ]:
# ==========================================
# 6. GRADIO TESTING DEMO
# ==========================================
def predict(img):
    if img is None: return None
    img_t = transform(img).unsqueeze(0).to(device)
    image_model.eval(); projection.eval()
    with torch.no_grad():
        features = image_model(img_t)
        image_features = F.normalize(projection(features), dim=-1)
        probs = (logit_scale.exp() * image_features @ text_embeddings.T).softmax(dim=-1).cpu().numpy()[0]

    top_5_idx = np.argsort(probs)[-5:][::-1]
    return {classes[i]: float(probs[i]) for i in top_5_idx}

gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=5),
    title="ViT AugReg Caltech-101 Classifier"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://17a201a013441d8c16.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
